# 3. Explainability: what the model looks at, and whether to believe it

Reads the outputs of `explain` and `lesions_report`. A plausible heatmap does not prove the model recognised a lesion, so the explanations are tested: a model-randomisation sanity check, and lesion probes against an ImageNet-only floor.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from drxai import GRADE_NAMES
from drxai.config import REPO_ROOT, cache_dir, load_config, resolve

# Read-only: this notebook loads files written by the pipeline commands and computes nothing that is reported elsewhere.
cfg = load_config(os.environ.get("DRXAI_CONFIG", REPO_ROOT / "configs" / "default.yaml"))
CACHE = cache_dir(cfg)
RUNS = resolve(cfg["paths"]["runs_dir"])

In [ ]:
comparison = json.loads((RUNS / "comparison.json").read_text())
PRIMARY = Path(comparison["primary"])
print("primary run (chosen on validation QWK):", PRIMARY.name)

In [ ]:
ex = json.loads((PRIMARY / "explain_summary.json").read_text())
print("Grad-CAM target:", ex["target"])

## Representative cases
Selected by fixed rules (highest confidence within each group), not chosen by eye.

In [ ]:
display(Image(str(PRIMARY / "figures" / "gradcam_gallery.png"), width=520))

## Triage case cards
Prediction and action, explanation, and clinical judgement kept in separate panels, including a confident error the referral rule cannot catch.

In [ ]:
display(Image(str(PRIMARY / "figures" / "case_cards.png"), width=1000))

## Sanity check (Adebayo et al., 2018)
If maps from the trained model and from a model with the explained block re-initialised are highly correlated, the maps reflect image structure rather than learned features.

In [ ]:
print(ex["sanity_check"])
display(Image(str(PRIMARY / "figures" / "sanity_check.png"), width=420))

## What the lesion flags allow
MMRDR gives seven present/absent lesion flags per image but no locations, so heatmaps cannot be scored spatially. Three flag-based analyses replace that: label consistency, a linear probe of lesion concepts in the model's features (against an ImageNet-only floor), and the lesion make-up of missed referable cases. Produced by `python -m drxai.lesions_report`.

In [ ]:
rep_md = PRIMARY / "lesions_report.md"
if rep_md.exists():
    display(Markdown(rep_md.read_text()))
    if (PRIMARY / "figures" / "lesion_probe.png").exists():
        display(Image(str(PRIMARY / "figures" / "lesion_probe.png"), width=720))
else:
    print("run python -m drxai.lesions_report --run", PRIMARY)
la_summary = ex["lesion_agreement"]
if isinstance(la_summary, dict):   # only when spatial lesion masks are configured
    d = la_summary["hit_minus_chance_95ci"]
    print(f"pointing hit rate minus chance: {d['point']:+.3f} (95% CI {d['lo']:+.3f} to {d['hi']:+.3f})")

## What this does and does not show
A lesion that a linear probe reads from the model's features well above the ImageNet-only floor is encoded by the model; it does not follow that the model uses it to grade, or that the heatmap points at it. A probe no better than the floor is reported as such rather than explained away.